In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. 加载数据集
data = pd.read_csv('D:/notebook文件/实验1/voice.csv')

# 2. 数据预处理
X = data.drop('label', axis=1)
y = data['label']

# 标签编码及映射关系保存
le = LabelEncoder()
y_encoded = le.fit_transform(y)
# 创建标签映射字典（用于后续还原性别字符串）
label_mapping = {0: le.inverse_transform([0])[0], 1: le.inverse_transform([1])[0]}
print(f"标签映射关系：{label_mapping}")  # 显示0/1对应male/female

# 3. 划分数据集
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    train_size=2000,
    test_size=1168,
    random_state=42,
    stratify=y_encoded
)

# 4. 训练模型
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# 5. 预测
y_pred = nb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# 6. 构建完整预测结果数据框
# 合并测试集特征、真实标签（编码+原始字符串）、预测标签（编码+原始字符串）
result_df = X_test.copy()
result_df['真实标签_编码'] = y_test
result_df['真实标签_性别'] = [label_mapping[code] for code in y_test]
result_df['预测标签_编码'] = y_pred
result_df['预测标签_性别'] = [label_mapping[code] for code in y_pred]
# 新增预测是否正确的标记
result_df['预测是否正确'] = (result_df['真实标签_编码'] == result_df['预测标签_编码']).map({True: '是', False: '否'})

# 7. 保存预测结果（CSV格式，可直接用Excel打开）
save_path = 'D:/notebook文件/实验1/voice_gender_prediction_results.csv'
result_df.to_csv(save_path, index=False, encoding='utf-8-sig')  # utf-8-sig支持中文显示

# 8. 输出关键信息
print(f"\n训练集规模：{X_train.shape[0]} 条")
print(f"测试集规模：{X_test.shape[0]} 条")
print(f"朴素贝叶斯模型预测准确度：{accuracy:.4f}")

print(f"\n预测结果已保存至：{save_path}")
print(f"\n结果文件包含列：{list(result_df.columns)}")

# 可选：输出预测结果统计
correct_count = sum(result_df['预测是否正确'] == '是')
incorrect_count = sum(result_df['预测是否正确'] == '否')
print(f"\n预测正确样本数：{correct_count} 条")
print(f"预测错误样本数：{incorrect_count} 条")

标签映射关系：{0: 'female', 1: 'male'}

训练集规模：2000 条
测试集规模：1168 条
朴素贝叶斯模型预测准确度：0.8818

预测结果已保存至：D:/notebook文件/实验1/voice_gender_prediction_results.csv

结果文件包含列：['meanfreq', 'sd', 'median', 'Q25', 'Q75', 'IQR', 'skew', 'kurt', 'sp.ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx', '真实标签_编码', '真实标签_性别', '预测标签_编码', '预测标签_性别', '预测是否正确']

预测正确样本数：1030 条
预测错误样本数：138 条
